# Fairness evaluation

We are going to carry out a fairness evaluation of different models on credit scoring scenarios, based on different datasets.

Several fairness and predictive metrics will be considered, as well as different fairness processing techniques.

**Datasets**

- German *(used in this notebook)*

- Homecredit

- Simulated data 

**Models**

- Logistic regression

- XGBoost

**Fairness metrics**

- Independence

- Sufficiency 

- Separation

**Predictive metrics**

- Accuracy

- Balanced Accuracy

- AUC

**Fairness processing techniques**

- Pre-processing

  - Reweighing

  - Disparate impact remover

- In-processing

  - Prejudice remover

  - Adversarial debasing 

  - Meta-fairness algorithm

- Post-processing

  - Reject option classification 

  - Group-wise Platt scaling

  - Equalized odds processor


**Evaluation schemas**

- *Simple*: one fairness processing method considered per evaluation schema.

- *Multi-processing*: multiple fairness processing methods are considered per evaluation schema.

- *Multi-sensitive*: multiple sensitive variables are considered per evaluation schema.

## Requirements

In [1]:
from aif360.datasets import GermanDataset
from aif360.metrics import ClassificationMetric
from aif360.sklearn.metrics import statistical_parity_difference
from aif360.sklearn.preprocessing import Reweighing, ReweighingMeta
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from PyMachineLearning.preprocessing import encoder, scaler, imputer

import numpy as np
import pandas as pd 
import random

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

pd.set_option('display.max_colwidth', None)

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'
pip install 'aif360[OptimalTransport]'
pip install 'aif360[FairAdapt]'


## Simple fairness evaluation 

### Data

In [2]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


### Response and predictors definition

In [3]:
##X = german_data_aif.features
##Y = german_data_aif.labels.ravel()

In [4]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


In [5]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [6]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute

### Outer evaluation method: simple-validation

In [28]:
# Outer evaluation: simple validation (train-test split)
##german_data_aif_train, german_data_aif_test = german_data_aif.split([0.85], shuffle=True, seed=123)

# Extract features and labels for train, validation, and test sets
##X_train = german_data_aif_train.features
##Y_train = german_data_aif_train.labels.ravel()

##X_test = german_data_aif_test.features
##Y_test = german_data_aif_test.labels.ravel()

In [138]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

# A_train = X_train[sens_variable_name] # sensitive variable / protected attribute in train set
# A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set

### Inner evaluation method: cross-validation

In [139]:
# Inner evaluation: stratified cross validation
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

---

#### HPO 

In [93]:
# Hyper-param optimization for the model using a predictive metric (balanced accuracy)

param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', None], 
    'C': [0.01, 0.1, 1, 10, 30, 50, 75, 100],  # Regularization strength
    'class_weight': ['balanced', None]
}

# Initialize the logistic regression model
logistic_reg = LogisticRegression(solver='liblinear', random_state=123)

# Set up RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=logistic_reg,
    param_distributions=param_grid,
    scoring='balanced_accuracy',
    cv=inner,
    n_iter=50
)

# Fit the model on the training data
random_search.fit(X_train, Y_train)

c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore th

RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=123, shuffle=True),
                   estimator=LogisticRegression(random_state=123,
                                                solver='liblinear'),
                   n_iter=50,
                   param_distributions={'C': [0.01, 0.1, 1, 10, 30, 50, 75,
                                              100],
                                        'class_weight': ['balanced', None],
                                        'penalty': ['l1', 'l2', 'elasticnet',
                                                    None]},
                   scoring='balanced_accuracy')

In [94]:
# Best hyper-params according to the optimization process
random_search.best_params_

{'penalty': 'l2', 'class_weight': 'balanced', 'C': 0.01}

In [95]:
# Validation value of the predictive metric (balanced accuracy)
random_search.best_score_

0.7142857142857144

In [96]:
logistic_reg.set_params(**random_search.best_params_)

LogisticRegression(C=0.01, class_weight='balanced', random_state=123,
                   solver='liblinear')

---

### Fairness metrics

In [97]:
logistic_reg.fit(X_train, Y_train)
Y_test_hat = logistic_reg.predict(X_test)

In [99]:
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set

In [100]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.251984126984127

`statistical_parity_difference`  $= Pr(\hat{Y} = \text{pos\_label} | D = \text{unprivileged}) - Pr(\hat{Y} = \text{pos\_label} | D = \text{privileged})$

- $\in [-1, 1]$

- The closer to $-1$, the more favorable are the results (predictions) of the models to the privileged class of the sensitive variable.

- The closer to $1$, the more favorable are the results (predictions) of the models to the unprivileged class of the sensitive variable.

- The closer to $0$, the more fairness (parity) in the model results (predictions).

- We can consider the metric in absolute value, then $\in [0,1]$, and the closer to $0$ the more fairness, the closer to $1$ the more parity difference (less fairness (?)).

**Doubts**

- No veo que esta metrica sea una medida de justicia en todos los casos. Ejemplo: concesion de creditos, variable sensible color de piel (blanco, negro). Imaginemos que el grupo de los negros reune una serie de condiciones que les hace objetivamente mas riesgosos que los blancos (peor situacion economica, mayores deudas etc). Que en este caso el modelo conceda significativamente mas creditos a los blancos que a los negros no seria injusto, no? Puede setar basado en criterios economicos objetivos que estan asociados a esos grupos, y no por el color de piel en si, no??


**TO DO: EXPLORING MORE METRICS**

https://aif360.readthedocs.io/en/latest/modules/generated/aif360.sklearn.metrics.statistical_parity_difference.html

---

### Computing fairness metric in the inner eval. process with cross validation

In [16]:
# Computing validation Fairness metrics using the same schema as for the predictive metric (Stratified KFold)

independence_iters, independence_2_iters, separation_iters, sufficiency_iters = [], [], [], []

for train_index, val_index in inner.split(X, Y):
    # Split the data into training and validation sets
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    Y_train, Y_val = Y.iloc[train_index], Y.iloc[val_index]
    A_train, A_val = X_train[sens_variable_name], X_val[sens_variable_name] # sensitive variable in train and val sets
    
    # Train logistic regression model
    logistic_reg.set_params(**random_search.best_params_)
    logistic_reg.fit(X_train, Y_train)

    # Predict on validation set
    Y_val_hat = logistic_reg.predict(X_val)
       
    # Convert the validation set to AIF360 BinaryLabelDataset format
    german_data_aif_val = german_data_aif.subset(val_index)  # Subset the original dataset using the validation indices
    german_data_aif_val_pred = german_data_aif_val.copy()
    german_data_aif_val_pred.labels = Y_val_hat.reshape(-1, 1)

    # Compute fairness metrics using AIF360
    metric_val = ClassificationMetric(german_data_aif_val, german_data_aif_val_pred, 
                                      unprivileged_groups=sens_privileged_groups, 
                                      privileged_groups=sens_unprivileged_groups)
    
    # Calculate fairness metrics
    independence_iters.append(np.abs(metric_val.statistical_parity_difference()))
    independence_2_iters.append(np.abs(statistical_parity_difference(
                                            y_true=Y_val,
                                            y_pred=Y_val_hat, 
                                            prot_attr=A_val,
                                            priv_group=sens_privileged_groups[0][sens_variable_name], 
                                            pos_label=response_favorable_label
                                            )
                                        ))
    separation_iters.append(np.abs(metric_val.false_positive_rate_difference() + metric_val.false_negative_rate_difference()) / 2)
    sufficiency_iters.append(np.abs(metric_val.positive_predictive_value(True) - metric_val.positive_predictive_value(False)))

# Compute average metrics across folds
avg_independence = np.mean(independence_iters)
avg_independence_2 = np.mean(independence_2_iters)
avg_separation = np.mean(separation_iters)
avg_sufficiency = np.mean(sufficiency_iters)

In [17]:
independence_iters

[0.1605244507441531,
 0.32455576964247485,
 0.2627545876184714,
 0.31568627450980397,
 0.21778584392014516]

In [18]:
independence_2_iters

[0.1605244507441531,
 0.32455576964247485,
 0.2627545876184714,
 0.31568627450980397,
 0.21778584392014516]

In [19]:
separation_iters

[0.2753322700691122,
 0.0010220125786163659,
 0.051587825352410105,
 0.10800000000000003,
 0.08584506828066546]

In [20]:
sufficiency_iters

[0.3548872180451128,
 0.17821782178217827,
 0.01927437641723362,
 0.06426332288401249,
 0.09259259259259256]

In [21]:
print(avg_independence, avg_independence_2, avg_separation, avg_sufficiency)

0.2562613852870097 0.2562613852870097 0.10435743525616084 0.14184706634422595


---

### CrossValScore for Fairness metrics

In [141]:
def cross_val_score_fairness(estimator, X, y, sens_variable_name, priv_group, pos_label, scoring, cv):
    
    # X must be a Pandas DataFrame
    if not isinstance(X, pd.DataFrame):
        raise TypeError('X must be a Pandas DataFrame')
    if isinstance(y, pd.Series):
        y = y.to_numpy()

    metric_iters = []
    # Split the data into training and validation sets 
    for train_index, val_index in cv.split(X, y): 
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        Y_train, Y_val = y[train_index], y[val_index]
        A_val = X_val[sens_variable_name] # sensitive variable in val set

        # Train logistic regression model
        estimator.fit(X_train, Y_train)

        # Predict on validation set
        Y_val_hat = estimator.predict(X_val)

        # Calculate fairness metrics for each iteration of the cross-validation process
        if scoring == 'statistical_parity_difference':
            metric_iters.append(statistical_parity_difference(y_true=Y_val, y_pred=Y_val_hat, prot_attr=A_val,
                                                              priv_group=priv_group, pos_label=pos_label))
        elif scoring == 'abs_statistical_parity_difference':
            metric_iters.append(np.abs(statistical_parity_difference(y_true=Y_val, y_pred=Y_val_hat, prot_attr=A_val,
                                                                     priv_group=priv_group, pos_label=pos_label)))
    
    # Compute the average of the metric along the iterations
    final_metric = np.mean(metric_iters)

    return final_metric, metric_iters

In [102]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

In [103]:
fairness_final_metric, fairness_metric_iters = cross_val_score_fairness(estimator=log_reg_estimator, X=X_train, y=Y_train, 
                                                                        sens_variable_name=sens_variable_name, 
                                                                        priv_group=sens_privileged_groups[0][sens_variable_name],
                                                                        pos_label=response_favorable_label, 
                                                                        scoring='abs_statistical_parity_difference', 
                                                                        cv=inner)

In [104]:
fairness_metric_iters

[0.3648504273504273,
 0.1523809523809524,
 0.13087606837606836,
 0.4318249038745933,
 0.18333333333333335]

In [56]:
fairness_final_metric

0.25265313706307496

In [105]:
predictive_metric_iters = cross_val_score(estimator=log_reg_estimator, X=X_train, y=Y_train, 
                                          scoring='balanced_accuracy', cv=inner)

In [106]:
predictive_metric_iters

array([0.68627451, 0.69187675, 0.65406162, 0.64005602, 0.65966387])

In [64]:
predictive_final_metric = np.mean(predictive_metric_iters)
predictive_final_metric

0.6663865546218487

**TO DO: Testing with more fairness metrics**

---

### RandomizedSearchCV for Fairness metrics

In [217]:
class RandomizedSearchCVFairness:
    
    def __init__(self, estimator, param_distributions, 
                 fairness_scoring, predictive_scoring, objective, 
                 fairness_scoring_direction, predictive_scoring_direction, 
                 fairness_weight, predictive_weight, 
                 cv, n_iter, random_state, sens_variable_name, priv_group, pos_label):

        self.estimator = estimator
        self.param_distributions = param_distributions
        self.fairness_scoring = fairness_scoring
        self.predictive_scoring = predictive_scoring
        self.objective = objective # ['fairness', 'predictive']
        self.fairness_scoring_direction = fairness_scoring_direction # ['maximize', ' minimize']
        self.predictive_scoring_direction = predictive_scoring_direction # ['maximize', ' minimize']
        self.fairness_weight = fairness_weight
        self.predictive_weight = predictive_weight
        self.cv = cv 
        self.n_iter = n_iter
        self.random_state = random_state
        self.sens_variable_name = sens_variable_name
        self.priv_group = priv_group
        self.pos_label = pos_label
        self.results_ = []

    def _random_param_sample(self):
        """Randomly sample a parameter combination from the distributions."""
        random_params = {key: random.choice(val) for key, val in self.param_distributions.items()}
        return random_params
    
    def fit(self, X, y):
        
        random.seed(self.random_state)
        for iter in range(self.n_iter):
            try:
                random_params = self._random_param_sample()
                self.estimator.set_params(**random_params)

                fairness_final_metric, _ = cross_val_score_fairness(estimator=self.estimator, X=X, y=y, 
                                                                    sens_variable_name=self.sens_variable_name, 
                                                                    priv_group=self.priv_group,
                                                                    pos_label=self.pos_label, 
                                                                    scoring=self.fairness_scoring, cv=self.cv)  
                  
                predictive_metric_iters = cross_val_score(estimator=self.estimator, X=X_train, y=Y_train, 
                                                          scoring=self.predictive_scoring, cv=inner)
                
                predictive_final_metric = np.mean(predictive_metric_iters)
                 
                self.results_.append({'params': random_params, 
                                      'predictive-score': predictive_final_metric, 
                                      'fairness-score': fairness_final_metric})
                
            except Exception as e:
                print(e)
            
            predictive_scores = [self.results_[i][f'predictive-score'] for i in range(len(self.results_))]
            fairness_scores = [self.results_[i][f'fairness-score'] for i in range(len(self.results_))]
            
            # Building the combined score
            scaler = MinMaxScaler(feature_range=(0,1))            
            predictive_scores_normalized = scaler.fit_transform(np.array(predictive_scores).reshape(-1, 1)).flatten()
            fairness_scores_normalized = scaler.fit_transform(np.array(fairness_scores).reshape(-1, 1)).flatten()
            if self.predictive_scoring_direction == 'minimize':
                predictive_scores_normalized = 1 - predictive_scores_normalized
            if self.fairness_scoring_direction == 'minimize':
                fairness_scores_normalized = 1 - fairness_scores_normalized
            combined_scores = predictive_scores_normalized * self.predictive_weight + fairness_scores_normalized * self.fairness_weight
            for i in range(len(self.results_)):
                self.results_[i]['combined-score'] = combined_scores[i]
            
            # Optimizing the parameters according to the objective and the scores
            # Obtaining the best params and score, and building a data-frame with the results
            score_list = [self.results_[i][f'{self.objective}-score'] for i in range(len(self.results_))]
            self.results_df_ = pd.DataFrame(self.results_)

            scoring_direction_map = {'combined': ('maximize', False),
                                     'fairness': (self.fairness_scoring_direction, None),
                                     'predictive': (self.predictive_scoring_direction, None)
                                    }
            scoring_direction, ascending_value = scoring_direction_map[self.objective]
            opt_function = np.argmax if scoring_direction == 'maximize' else np.argmin
            ascending_value = False if scoring_direction == 'maximize' else True 
            best_score_idx = opt_function(score_list)
            self.results_df_ = self.results_df_.sort_values(by=f'{self.objective}-score', ascending=ascending_value)
            self.best_params_ = self.results_[best_score_idx]['params']
            self.best_score_ = self.results_[best_score_idx][f'{self.objective}-score']

    def best_params(self):
        return self.best_params_

    def best_score(self):
        return self.best_score_

    def results(self):
        return self.results_df_

In [218]:
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet', None], 
    'C': [0.01, 0.1, 1, 10, 30, 50, 75, 100],  # Regularization strength
    'class_weight': ['balanced', None]
}

In [219]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                           fairness_scoring='abs_statistical_parity_difference', 
                                           predictive_scoring='balanced_accuracy',
                                           objective='fairness', 
                                           fairness_scoring_direction='minimize',
                                           predictive_scoring_direction='maximize',
                                           fairness_weight=0.5, predictive_weight=0.5,
                                           cv=inner, n_iter=15, random_state=123,
                                           sens_variable_name=sens_variable_name, 
                                           priv_group=sens_privileged_groups[0][sens_variable_name],
                                           pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

fairness_random_search.results()

c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


,params,predictive-score,fairness-score,combined-score
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None}",0.500000,0.000000,0.500000
6,"{'penalty': 'l1', 'C': 30, 'class_weight': None}",0.670868,0.251893,0.433774
7,"{'penalty': 'l1', 'C': 30, 'class_weight': None}",0.670868,0.251893,0.433774
1,"{'penalty': 'l1', 'C': 75, 'class_weight': None}",0.671709,0.253282,0.433167
8,"{'penalty': 'l1', 'C': 75, 'class_weight': None}",0.671709,0.253282,0.433167
0,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced'}",0.710644,0.255278,0.521847
5,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced'}",0.708123,0.259195,0.508528
2,"{'penalty': 'l2', 'C': 50, 'class_weight': None}",0.662185,0.264123,0.390255
3,"{'penalty': 'l2', 'C': 1, 'class_weight': 'balanced'}",0.707283,0.266942,0.492021


In [210]:
fairness_random_search.best_params()

{'penalty': 'l1', 'C': 0.01, 'class_weight': None}

In [211]:
fairness_random_search.best_score()

0.0

In [220]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                           fairness_scoring='abs_statistical_parity_difference', 
                                           predictive_scoring='balanced_accuracy',
                                           objective='predictive', 
                                           fairness_scoring_direction='minimize',
                                           predictive_scoring_direction='maximize',
                                           fairness_weight=0.5, predictive_weight=0.5,
                                           cv=inner, n_iter=15, random_state=123,
                                           sens_variable_name=sens_variable_name, 
                                           priv_group=sens_privileged_groups[0][sens_variable_name],
                                           pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

fairness_random_search.results()

c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Only 'saga' solver supports elasticnet penalty, got solver=liblinear.


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


,params,predictive-score,fairness-score,combined-score
0,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced'}",0.710644,0.255278,0.521847
5,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced'}",0.708123,0.259195,0.508528
3,"{'penalty': 'l2', 'C': 1, 'class_weight': 'balanced'}",0.707283,0.266942,0.492021
1,"{'penalty': 'l1', 'C': 75, 'class_weight': None}",0.671709,0.253282,0.433167
8,"{'penalty': 'l1', 'C': 75, 'class_weight': None}",0.671709,0.253282,0.433167
6,"{'penalty': 'l1', 'C': 30, 'class_weight': None}",0.670868,0.251893,0.433774
7,"{'penalty': 'l1', 'C': 30, 'class_weight': None}",0.670868,0.251893,0.433774
2,"{'penalty': 'l2', 'C': 50, 'class_weight': None}",0.662185,0.264123,0.390255
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None}",0.500000,0.000000,0.500000


In [221]:
fairness_random_search = RandomizedSearchCVFairness(estimator=log_reg_estimator, param_distributions=param_grid, 
                                           fairness_scoring='abs_statistical_parity_difference', 
                                           predictive_scoring='balanced_accuracy',
                                           objective='combined', 
                                           fairness_scoring_direction='minimize',
                                           predictive_scoring_direction='maximize',
                                           fairness_weight=0.5, predictive_weight=0.5,
                                           cv=inner, n_iter=15, random_state=123,
                                           sens_variable_name=sens_variable_name, 
                                           priv_group=sens_privileged_groups[0][sens_variable_name],
                                           pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

fairness_random_search.results()

c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


,params,predictive-score,fairness-score,combined-score
0,"{'penalty': 'l1', 'C': 30, 'class_weight': 'balanced'}",0.710644,0.255278,0.521847
5,"{'penalty': 'l1', 'C': 1, 'class_weight': 'balanced'}",0.708123,0.259195,0.508528
4,"{'penalty': 'l1', 'C': 0.01, 'class_weight': None}",0.500000,0.000000,0.500000
3,"{'penalty': 'l2', 'C': 1, 'class_weight': 'balanced'}",0.707283,0.266942,0.492021
6,"{'penalty': 'l1', 'C': 30, 'class_weight': None}",0.670868,0.251893,0.433774
7,"{'penalty': 'l1', 'C': 30, 'class_weight': None}",0.670868,0.251893,0.433774
1,"{'penalty': 'l1', 'C': 75, 'class_weight': None}",0.671709,0.253282,0.433167
8,"{'penalty': 'l1', 'C': 75, 'class_weight': None}",0.671709,0.253282,0.433167
2,"{'penalty': 'l2', 'C': 50, 'class_weight': None}",0.662185,0.264123,0.390255


---

### Fairness processing methods

#### Pre-processing

##### Reweighing

In [174]:
from sklearn.base import BaseEstimator, ClassifierMixin

class ReweighingMetaEstimator(BaseEstimator, ClassifierMixin):
   
    def __init__(self, estimator, prot_attr):
        
        self.estimator = estimator
        self.prot_attr = prot_attr

    def fit(self, X, y):
        
        reweigher_ = Reweighing(prot_attr=X[self.prot_attr])
        self.meta_estimator = ReweighingMeta(estimator=self.estimator, reweigher=reweigher_)
        self.meta_estimator.fit(X, y)
        self.classes_ = self.meta_estimator.classes_

        return self
    
    def predict(self, X):

        return self.meta_estimator.predict(X)
    
    def predict_proba(self, X):

        return self.meta_estimator.predict_proba(X)

In [164]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

In [165]:
fair_model = meta = ReweighingMetaEstimator(estimator=LogisticRegression(solver='liblinear', random_state=123), 
                               prot_attr=sens_variable_name)

fair_model.fit(X=X_train, y=Y_train)

ReweighingMetaEstimator(estimator=LogisticRegression(random_state=123,
                                                     solver='liblinear'),
                        prot_attr='age')

In [166]:
Y_test_hat_re = fair_model.predict(X_test)
Y_test_hat_re

array([1., 1., 1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 0., 0., 0., 1., 1.,
       1., 1., 1., 0., 0., 0., 1., 0., 1., 1., 1., 0., 0., 0., 1., 1., 1.,
       0., 1., 1., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
       0., 1., 1., 1., 0., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1., 1.,
       0., 1., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.,
       1., 0., 1., 0., 1., 1., 0., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 1.])

In [167]:
fair_model.predict_proba(X_test)

array([[0.05582486, 0.94417514],
       [0.23913795, 0.76086205],
       [0.0940255 , 0.9059745 ],
       [0.10271948, 0.89728052],
       [0.23557883, 0.76442117],
       [0.64635807, 0.35364193],
       [0.53352253, 0.46647747],
       [0.308247  , 0.691753  ],
       [0.04581351, 0.95418649],
       [0.83554819, 0.16445181],
       [0.21149137, 0.78850863],
       [0.51488785, 0.48511215],
       [0.64706916, 0.35293084],
       [0.53833336, 0.46166664],
       [0.58429157, 0.41570843],
       [0.23709134, 0.76290866],
       [0.19169264, 0.80830736],
       [0.19353593, 0.80646407],
       [0.07604838, 0.92395162],
       [0.20489348, 0.79510652],
       [0.69822276, 0.30177724],
       [0.63593547, 0.36406453],
       [0.60687088, 0.39312912],
       [0.0694049 , 0.9305951 ],
       [0.92467953, 0.07532047],
       [0.08723186, 0.91276814],
       [0.04349367, 0.95650633],
       [0.1631391 , 0.8368609 ],
       [0.52223112, 0.47776888],
       [0.51871276, 0.48128724],
       [0.

In [89]:
balanced_accuracy_score(y_pred=Y_test_hat_re, y_true=Y_test)

0.6396825396825396

In [90]:
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set

statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat_re, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.17658730158730163

In [91]:
log_reg_estimator.fit(X_train, Y_train)
Y_test_hat = log_reg_estimator.predict(X_test)

In [92]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6396825396825396

In [93]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.32539682539682535

**TO DO: testing more pre-processing methods**

#### In-processing

**TO DO: testing more in-processing methods**

#### Post-processing

**TO DO: testing more post-processing methods**

---

Probar lo anterior con sk pipelines y con pipelines que involucren fairnes processing

In [144]:
from sklearn.base import BaseEstimator, TransformerMixin

class ToPandasTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        """
        Transformer to convert NumPy array back to pandas DataFrame.
        
        Parameters:
        - columns: The list of column names to assign to the DataFrame
        """
        self.columns = columns

    def fit(self, X, y=None):
        """
        No fitting necessary for this transformer.
        """
        return self

    def transform(self, X):
        """
        Convert the NumPy array X to a pandas DataFrame with the given columns.
        
        Parameters:
        - X: NumPy array to convert to DataFrame
        
        Returns:
        - pandas DataFrame with the specified column names
        """
        return pd.DataFrame(X, columns=self.columns)


In [175]:
quant_pipeline = Pipeline([
    ('imputer', imputer()),
    ('scaler', scaler())
    ])

cat_pipeline = Pipeline([
    ('imputer', imputer()),
    ('encoder', encoder())
    ])

quant_cat_processing = ColumnTransformer(transformers=[('quant', quant_pipeline, quant_predictors),
                                                       ('cat', cat_pipeline, cat_predictors)])

 
models, pipelines, param_grid = {}, {}, {}

models['knn'] = KNeighborsClassifier(n_jobs=-1)
models['trees'] = DecisionTreeClassifier(random_state=123), 
models['extra_trees'] = ExtraTreesClassifier(random_state=123),
models['RF'] = RandomForestClassifier(random_state=123), 
models['HGB'] = HistGradientBoostingClassifier(random_state=123), 
models['NN'] = MLPClassifier(random_state=123),
models['SVM'] = LinearSVC(random_state=123),  
models['XGB'] = XGBClassifier(random_state=123),
models['logistic_reg'] = LogisticRegression(solver='liblinear', random_state=123),
models['logistic_reg_reweighing'] = ReweighingMetaEstimator(estimator=LogisticRegression(solver='liblinear', random_state=123), 
                                                            prot_attr=sens_variable_name)


for key, model in models.items():

    if key == 'logistic_reg_reweighing':

        pipelines[key] = Pipeline([
                ('preprocessing', quant_cat_processing),
                ('to_pandas', ToPandasTransformer(columns=predictors)), # ReweighingMetaEstimator need a Pandas df X as input to read the sens_variable_name 
                (key, model) 
                ])            

    else:

        pipelines[key] = Pipeline([
                ('preprocessing', quant_cat_processing),
                (key, model) 
                ])

In [168]:
preprocessing_grid = {'preprocessing__quant__scaler__apply': [True, False],
                      'preprocessing__quant__scaler__method': ['standard', 'min-max'],
                      'preprocessing__cat__encoder__method': ['ordinal', 'one-hot'],
                      'preprocessing__cat__imputer__apply':  [False],
                      'preprocessing__quant__imputer__apply': [False]
                    }

In [169]:
param_grid['knn'] = preprocessing_grid.copy()
param_grid['knn'].update({'knn__n_neighbors': np.arange(1,20),
                          'knn__metric': ['cosine', 'minkowski', 'cityblock'],
                          'knn__p': np.arange(1,4)
                        })

In [177]:
param_grid['logistic_reg_reweighing'] = preprocessing_grid.copy()
param_grid['logistic_reg_reweighing'].update({'logistic_reg_reweighing__estimator__penalty': ['l1', 'l2', 'elasticnet', None],
                                              'logistic_reg_reweighing__estimator__C':  [0.01, 0.1, 1, 10, 30, 50, 75, 100],
                                              'logistic_reg_reweighing__estimator__class_weight': ['balanced', None]
                                            })

In [171]:
fairness_random_search = RandomizedSearchCVFairness(estimator=pipelines['knn'], 
                                                    param_distributions=param_grid['knn'], 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='fairness', direction='minimize',
                                                    cv=inner, n_iter=15, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

In [172]:
fairness_random_search.results()

,params,predictive-score,fairness-score
7,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 3, 'knn__metric': 'cityblock', 'knn__p': 2}",0.541176,0.030592
5,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 17, 'knn__metric': 'minkowski', 'knn__p': 3}",0.530532,0.046757
1,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 11, 'knn__metric': 'cityblock', 'knn__p': 2}",0.550980,0.047427
6,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 3, 'knn__metric': 'minkowski', 'knn__p': 3}",0.544258,0.052172
3,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 5, 'knn__metric': 'cosine', 'knn__p': 1}",0.611765,0.052401
11,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 9, 'knn__metric': 'minkowski', 'knn__p': 3}",0.540616,0.059850
8,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 12, 'knn__metric': 'cosine', 'knn__p': 2}",0.592437,0.067009
12,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 3, 'knn__metric': 'cosine', 'knn__p': 2}",0.583754,0.072868
4,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 2, 'knn__metric': 'minkowski', 'knn__p': 2}",0.507563,0.082632
13,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'knn__n_neighbors': 11, 'knn__metric': 'minkowski', 'knn__p': 2}",0.608683,0.150370


In [196]:
fairness_random_search = RandomizedSearchCVFairness(estimator=pipelines['logistic_reg_reweighing'], 
                                                    param_distributions=param_grid['logistic_reg_reweighing'], 
                                                    fairness_scoring='abs_statistical_parity_difference', 
                                                    predictive_scoring='balanced_accuracy',
                                                    objective='fairness', direction='minimize',
                                                    cv=inner, n_iter=15, random_state=123,
                                                    sens_variable_name=sens_variable_name, 
                                                    priv_group=sens_privileged_groups[0][sens_variable_name],
                                                    pos_label=response_favorable_label)

fairness_random_search.fit(X=X_train, y=Y_train)

Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False


c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(
c:\Users\fscielzo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1193: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  warnings.warn(


Unsupported set of arguments: The combination of penalty='None' and loss='logistic_regression' is not supported, Parameters: penalty=None, loss='logistic_regression', dual=False
Only 'saga' solver supports elasticnet penalty, got solver=liblinear.
Only 'saga' solver supports elasticnet penalty, got solver=liblinear.


In [197]:
results_df = fairness_random_search.results()
results_df

,params,predictive-score,fairness-score
0,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'min-max', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 0.01, 'logistic_reg_reweighing__estimator__class_weight': None}",0.500000,0.000000
1,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 0.01, 'logistic_reg_reweighing__estimator__class_weight': None}",0.500000,0.000000
2,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l2', 'logistic_reg_reweighing__estimator__C': 0.01, 'logistic_reg_reweighing__estimator__class_weight': None}",0.558543,0.037999
3,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'ordinal', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l2', 'logistic_reg_reweighing__estimator__C': 50, 'logistic_reg_reweighing__estimator__class_weight': None}",0.668908,0.107687
8,"{'preprocessing__quant__scaler__apply': True, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 75, 'logistic_reg_reweighing__estimator__class_weight': None}",0.666947,0.109075
4,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 10, 'logistic_reg_reweighing__estimator__class_weight': 'balanced'}",0.687955,0.115154
6,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 50, 'logistic_reg_reweighing__estimator__class_weight': None}",0.666947,0.118157
5,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 30, 'logistic_reg_reweighing__estimator__class_weight': None}",0.666106,0.125849
7,"{'preprocessing__quant__scaler__apply': False, 'preprocessing__quant__scaler__method': 'standard', 'preprocessing__cat__encoder__method': 'one-hot', 'preprocessing__cat__imputer__apply': False, 'preprocessing__quant__imputer__apply': False, 'logistic_reg_reweighing__estimator__penalty': 'l1', 'logistic_reg_reweighing__estimator__C': 1, 'logistic_reg_reweighing__estimator__class_weight': None}",0.670028,0.128230


---

**Classic vs Fair ML Evaluation schemas** 

Inner evaluation 

 - predictive metric (bal acc) y fairness metrics are computed
 
 - best pipeline is selected based on ???
     - In classic ML --> predictive metric
     - In fair ML --> a new metric defined as a function of both the predictive and fairness metrics (normalize all the metrics (bring all them to [0,1], for example) and compute the average, or sth like that) ???

Outer evaluation

   - Estimation of future performance based on ???

     - In classic ML --> predictive metric
     - In fair ML --> that new metric composed by both predictive and fairness metrics ???

Would be interesting adding in PyFairnesAI functionalities for computing both the fairness metrics and the new final metric in the inner evaluation allowing cross-validation methods. 

Basically imitate PyMachineLearning but focus on the fairness metrics.